# 12.08 · Optional reference: LangSmith Deployment and the Agent Server

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lessons 07.06 (`langgraph dev`), 12.04–12.05 (our service, its auth). |
| 🔑 Keys | none (everything runs on a local `langgraph dev` with the offline desk) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

> 📎 **Optional reference lesson.** Module 12 ships our own FastAPI service. This lesson shows the managed path: LangGraph's **Agent Server** (what LangSmith Deployment runs), so you can choose between them with numbers. Nothing here is needed for 12.09 or the project.

## Learning objectives

1. **Map** the Agent Server's resources (assistants, threads, runs, crons) onto what we built by hand in 12.04–12.05.
2. **Serve** the support desk with `langgraph.json` and `langgraph dev`, and call it with the SDK.
3. **Secure** it with custom auth: owner filters, server-set context, and who may answer an approval.
4. **Choose** a double-texting strategy (reject, enqueue, interrupt, rollback) by watching each one.
5. **Use** background runs, crons, RemoteGraph and the `/mcp` endpoint, and know why webhooks to localhost are blocked.
6. **Compare** features and cost with our own service, and decide what 12.09 ships.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Agent Server](https://docs.langchain.com/langsmith/agent-server) | What the server provides: assistants, threads, runs, task queue, persistence |
| [Application structure](https://docs.langchain.com/langsmith/application-structure) | `langgraph.json`: `dependencies`, `graphs`, `env`, `auth` |
| [LangGraph CLI](https://docs.langchain.com/langsmith/cli) | `langgraph dev`, `build`, `up` |
| [Custom auth](https://docs.langchain.com/langsmith/custom-auth) | `@auth.authenticate`, `@auth.on`, filters by metadata |
| [Double texting](https://docs.langchain.com/langsmith/double-texting) | reject, enqueue (default), interrupt, rollback |
| [Background runs](https://docs.langchain.com/langsmith/background-run) · [Cron jobs](https://docs.langchain.com/langsmith/cron-jobs) · [Webhooks](https://docs.langchain.com/langsmith/use-webhooks) | Runs without a waiting client |
| [RemoteGraph](https://docs.langchain.com/langsmith/use-remote-graph) · [MCP endpoint](https://docs.langchain.com/langsmith/server-mcp) | Calling a deployed graph from code or from an MCP client |
| [Deploy to cloud](https://docs.langchain.com/langsmith/deploy-to-cloud) · [Standalone server](https://docs.langchain.com/langsmith/deploy-standalone-server) | Hosting options and what each needs |
| [LangSmith pricing](https://www.langchain.com/pricing) | Plans, the free deployment, metering rates |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-cli[inmem]>=0.4.35,<1" "langgraph-api>=0.16.1,<1" "langgraph-sdk>=0.4.7,<1" "pydantic-settings>=2.16.0,<3" "httpx>=0.28.1,<1" "pyjwt[crypto]>=2.15.1,<3"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph-cli", "langgraph-api", "langgraph-sdk", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-agent-server'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · The same desk, two servers

| What we built (12.04–12.05) | Agent Server equivalent |
|---|---|
| `POST /threads`, owner in our store | **Threads**, owner in thread metadata (via auth) |
| `RunManager`: queue, limits, cancel | **Runs** on a built-in task queue, with **double-texting** strategies |
| `/runs/{id}/stream` (SSE) | `runs.stream(...)`, `stream_mode="messages-tuple"` and others |
| Approvals: `GET /approvals`, `POST /resume` | The thread is `interrupted`; a run with `command={"resume": …}` |
| Postgres checkpointer | Built in (in memory under `langgraph dev`, Postgres in a deployment) |
| `auth.py` (API keys, JWT) | `Auth()` handlers in `auth.py`, named in `langgraph.json` |
| — | **Assistants** (versioned configs of a graph), **crons**, **webhooks**, `/mcp`, Studio |

The app is in `apps/m12_agent_server/`: `graph.py` builds the same handoffs desk (offline model by default, `MASTERY_SERVICE_MODE=live` for Groq), `auth.py` reuses `ServiceSettings`' callers, and `langgraph.json` ties them together.

In [ ]:
# 🧪 Offline: the deployment manifest.
import json

APP_DIR = REPO_ROOT / "apps" / "m12_agent_server"
print(json.dumps(json.loads((APP_DIR / "langgraph.json").read_text(encoding="utf-8")), indent=2))

**You should now see** `dependencies: ["../.."]` (install the course package from the repo root), one graph `support_desk` at `./graph.py:graph`, `auth` at `./auth.py:auth`, and `env: ../../.env`. The graph is compiled **without** a checkpointer: the server brings its own.

---
## Part 2 · Start `langgraph dev`

`DevServer` (from Lesson 07.06) runs `langgraph dev --no-browser --no-reload` as a child process, waits for `/ok`, and stops it at the end. `MASTERY_DEMO_DELAY_S=0.5` makes each offline model call take half a second, so runs last long enough to double-text.

> In a terminal you would run `cd apps/m12_agent_server` and `langgraph dev`; it prints the API URL and a Studio link.

In [ ]:
# 🧪 Offline: start a local Agent Server for this notebook (stopped in the last Part).
import atexit

from langgraph_sdk import get_client

from mastery.graphs.server import DevServer

OUT = REPO_ROOT / "data" / "outputs" / "m12"
srv = DevServer(APP_DIR, env={"MASTERY_DEMO_DELAY_S": "0.5", "MASTERY_SERVICE_MODE": "demo"},
                log_path=OUT / "agent-server.log").start()
atexit.register(srv.stop)
customer = get_client(url=srv.url, headers={"Authorization": "Bearer demo-customer"})
other = get_client(url=srv.url, headers={"Authorization": "Bearer demo-other"})
manager = get_client(url=srv.url, headers={"Authorization": "Bearer demo-manager"})
anonymous = get_client(url=srv.url)
print("API   :", srv.url)
print("Studio:", srv.studio_url)
print("graphs:", [a["graph_id"] for a in await customer.assistants.search()])

**You should now see** the local URL, a Studio link (open it in a browser to see the graph; Studio needs a LangSmith login), and `graphs: ['support_desk']`: the server created a default **assistant** for each graph in `langgraph.json`.

---
## Part 3 · Custom auth: who may see and do what

`auth.py` has four handlers:

- `@auth.authenticate` turns `Authorization: Bearer …` into a user (demo keys, or JWTs as in 12.05), or answers **401**.
- `@auth.on` (every resource): stamps `metadata.owner` on create and returns the filter `{"owner": <id>}` on every read; managers get no filter.
- `@auth.on.threads.create_run`: a customer's run gets `context={"customer_id": <their id>}` set **by the server**, and a customer may **not** send `command={"resume": …}`.
- `@auth.on.assistants`: only managers may create, change or delete assistants.

In [ ]:
# 🧪 Offline: 401 without a key, owner metadata, 404 for someone else's thread.
from langgraph_sdk.errors import APIStatusError

async def status_of(call):
    try:
        await call
        return 200
    except APIStatusError as exc:
        return exc.response.status_code

print("anonymous create :", await status_of(anonymous.threads.create()))
thread = await customer.threads.create()
print("thread metadata  :", thread["metadata"])
print("other customer   :", await status_of(other.threads.get(thread["thread_id"])),
      "· their search:", len(await other.threads.search()))
print("manager sees it  :", await status_of(manager.threads.get(thread["thread_id"])))
print("customer assistant create:", await status_of(customer.assistants.create("support_desk")))

run = await customer.runs.create(thread["thread_id"], "support_desk",
                                 input={"messages": [{"role": "user", "content": "I'm C-1001, show my invoices"}]},
                                 context={"customer_id": "C-1001"})        # tries to act as someone else
result = await customer.runs.join(thread["thread_id"], run["run_id"])
print("answer           :", result["messages"][-1]["content"][:110])

### What just happened?

- No key → **401**; another customer's `get` → **404** (it doesn't exist *for them*) and their search is empty. That is the `{"owner": …}` filter.
- The signed-in customer is `C-1002` (the `demo-customer` key). They typed *"I'm C-1001"* and also sent `context={"customer_id": "C-1001"}`. `act_as_owner` overwrote the context with `C-1002`, so the invoice tool refused to act on C-1001 (the 12.05 `SupportContext` check). Never trust identity fields in a request body.
- Only managers may create assistants (**403**).

---
## Part 4 · Approvals: the interrupt belongs to the manager

A refund above the threshold calls `interrupt()`; the run ends and the thread's status becomes `interrupted`. Anyone who can create a run on the thread could resume it with `command={"resume": "approve"}`. With an owner-only filter, that includes **the customer asking for the refund**. `act_as_owner` refuses that with **403**; the manager (no filter) answers.

In [ ]:
# 🧪 Offline: a refund pauses for approval; the customer can't approve it, the manager can.
REFUND = "I'm customer C-1002. I want a refund of $199 for INV-2001, we moved to another vendor."
t2 = await customer.threads.create()
r = await customer.runs.create(t2["thread_id"], "support_desk", input={"messages": [{"role": "user", "content": REFUND}]})
await customer.runs.join(t2["thread_id"], r["run_id"])
state = await customer.threads.get_state(t2["thread_id"])
print("thread status   :", (await customer.threads.get(t2["thread_id"]))["status"])
print("pending approval:", {k: v for k, v in state["interrupts"][0]["value"].items() if k in ("invoice_id", "amount", "max_amount")})
print("customer resume :", await status_of(customer.runs.create(t2["thread_id"], "support_desk", command={"resume": "approve"})))
rr = await manager.runs.create(t2["thread_id"], "support_desk", command={"resume": "approve"})
out = await manager.runs.join(t2["thread_id"], rr["run_id"])
print("after approval  :", out["messages"][-1]["content"][:100])

**You should now see** `interrupted`, the pending invoice and amount, `customer resume : 403`, then the refund confirmation. Without the `command` check the customer would get 200 and approve their own refund: test your auth with the *attacker's* key, not only the happy path.

---
## Part 5 · Double texting

A user sends a second message while the first run is still working. The server lets you choose per run (`multitask_strategy=`). We start a run, wait 1.2 s (two or three offline model steps), and send a second message with each strategy.

In [ ]:
# 🧪 Offline: the four strategies, measured.
import asyncio

FIRST = {"messages": [{"role": "user", "content": "How do I rotate an API key?"}]}
SECOND = {"messages": [{"role": "user", "content": "Actually, show my invoices"}]}
rows = []
for strategy in ["reject", "enqueue", "interrupt", "rollback"]:
    t = await customer.threads.create()
    a = await customer.runs.create(t["thread_id"], "support_desk", input=FIRST)
    await asyncio.sleep(1.2)
    try:
        b = await customer.runs.create(t["thread_id"], "support_desk", input=SECOND, multitask_strategy=strategy)
    except APIStatusError as exc:
        await customer.runs.join(t["thread_id"], a["run_id"])
        rows.append({"strategy": strategy, "second run": f"HTTP {exc.response.status_code}", "first run": "success", "user messages kept": 1})
        continue
    await customer.runs.join(t["thread_id"], b["run_id"])
    runs = {x["run_id"]: x["status"] for x in await customer.runs.list(t["thread_id"])}
    values = (await customer.threads.get_state(t["thread_id"]))["values"]
    humans = [m for m in values["messages"] if m["type"] == "human"]
    rows.append({"strategy": strategy, "second run": runs[b["run_id"]],
                 "first run": runs.get(a["run_id"], "deleted"), "user messages kept": len(humans)})
table(rows, ["strategy", "second run", "first run", "user messages kept"])

### What just happened?

| Strategy | Behaviour we measured | Use it when |
|---|---|---|
| `reject` | Second request gets **409**; the first finishes | One run at a time is a hard rule (payments) |
| `enqueue` (default) | Both run, in order; 2 user messages | Chat where every message matters |
| `interrupt` | First is `interrupted`, its progress kept; 2 user messages | The user corrected themselves mid-answer |
| `rollback` | First run **deleted** with its input; 1 user message | The second message replaces the first |

With `interrupt`, a tool call may be half-done when the run stops; the docs warn your graph must handle that. Our 12.04 `RunManager` behaves like `reject` per thread (409), which is why the desk UI disables *Send* while a run is active.

---
## Part 6 · Runs without a waiting client: background, stateless, cron, webhooks

`runs.create` already was a **background run** (we joined it later). Two more shapes: a **stateless** run (`thread_id=None`, nothing kept), and a **cron** that creates runs on a schedule.

In [ ]:
# 🧪 Offline: a stateless streamed run, a cron, and a webhook that the server refuses.
events = {}
async for part in customer.runs.stream(None, "support_desk", input=FIRST, stream_mode="messages-tuple"):
    events[part.event] = events.get(part.event, 0) + 1
print("stateless stream events:", events)

cron = await customer.crons.create("support_desk", schedule="0 8 * * 1",
                                   input={"messages": [{"role": "user", "content": "Weekly: any open tickets for me?"}]})
print("cron                   :", cron["schedule"], "· next run", str(cron.get("next_run_date"))[:16])
await customer.crons.delete(cron["cron_id"])

t3 = await customer.threads.create()
try:
    await customer.runs.create(t3["thread_id"], "support_desk", input=FIRST, webhook="http://127.0.0.1:9/done")
except APIStatusError as exc:
    print("webhook to 127.0.0.1   :", exc.response.status_code, "·", str(exc).splitlines()[0][-40:])

### What just happened?

- The stateless stream sent one `metadata` event and a few `messages` events; no thread was stored.
- The cron was stored with its schedule (Mondays 08:00 UTC) and deleted again. In a deployment each tick creates a run.
- The **webhook** to `127.0.0.1` was refused with **422** (*loopback IP range*). Webhooks are server-side HTTP requests to a URL the *client* chooses: without a block, any caller could make the server call its own internal endpoints (SSRF). In production configure `"webhooks": {"url": {"allowed_domains": ["hooks.example.com"], "require_https": true}}` in `langgraph.json` and verify the callback on your side.

---
## Part 7 · Call the deployment from code: RemoteGraph and MCP

`RemoteGraph` makes a deployed graph look like a local one: `invoke`, `stream`, and you can add it as a node in another graph. The `/mcp` endpoint (Streamable HTTP) exposes each assistant as an **MCP tool**, so any MCP client (Lessons 05.02–05.03) can call the desk.

In [ ]:
# 🧪 Offline: RemoteGraph.ainvoke and MCP tools/list against the same server.
import httpx
from langgraph.pregel.remote import RemoteGraph

remote = RemoteGraph("support_desk", url=srv.url, headers={"Authorization": "Bearer demo-customer"})
answer = await remote.ainvoke(FIRST)
print("RemoteGraph:", answer["messages"][-1]["content"][:80])

resp = httpx.post(f"{srv.url}/mcp", json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}},
                  headers={"Authorization": "Bearer demo-customer", "Accept": "application/json, text/event-stream"})
tools = resp.json()["result"]["tools"]
print("MCP tools  :", resp.status_code, [t["name"] for t in tools])

**You should now see** the FAQ answer through `RemoteGraph`, and `MCP tools : 200 ['support_desk']`. The MCP endpoint uses the same auth handlers: without the header it answers 401.

---
## Part 8 · Hosting options and cost (checked 2026-10-16)

| Option | What runs where | Needs | Cost for this course |
|---|---|---|---|
| `langgraph dev` | Your PC, in memory | `langgraph-cli[inmem]` | Free; **development only** (no persistence across restarts) |
| LangSmith **Cloud** (serverless or dedicated) | LangChain's cloud, from a GitHub repo | **Plus** plan ($39/seat/month) | Plus includes 1 free small serverless deployment; more are metered (runtime 0.0675 LSU per vCPU-hour, 1 LSU = $1) |
| **Standalone / self-hosted** Agent Server | Your Docker/Kubernetes, with Postgres and Redis | `LANGGRAPH_CLOUD_LICENSE_KEY` (Enterprise) and `LANGSMITH_API_KEY` | Enterprise pricing |
| **Our FastAPI service** (12.04–12.07) | Any container host | Docker | The host's free tier; you maintain the code |

[**Agent Chat UI**](https://docs.langchain.com/oss/python/langchain/ui) (`npx create-agent-chat-app`, or the hosted agentchat.vercel.app) is a ready chat front end for any Agent Server URL; it understands interrupts.

### Decision for 12.09

On a free budget, **12.09 ships our own service** (`apps/m12_service`): it runs on any free container host, we already have its tests, metrics and runbook, and nothing needs a paid plan. The Agent Server is the better choice when a team pays for Plus and wants crons, double texting, Studio and MCP without writing them. Because both serve the same `build_handoffs_desk` graph, moving later is a deployment change, not a rewrite.

In [ ]:
# 🧪 Offline: stop the local server.
srv.stop()
print("stopped:", srv.url)

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `langgraph dev` exits at once | Missing `langgraph-cli[inmem]`, or an import error in `graph.py` / `auth.py` | Read `data/outputs/m12/agent-server.log`; run `python graph.py` from the app folder |
| Every call answers 401 | No `Authorization` header, or `MASTERY_SERVICE_API_KEYS` in `.env` replaced the demo keys | Send `Bearer <key>` from your own key list |
| A customer approves their own refund | Owner-only auth lets the owner resume the interrupt | Check `command` in `@auth.on.threads.create_run`; only approvers may resume |
| The tool acts on the customer typed in the chat | Identity taken from the request body or the message | Set `context` on the server from `ctx.user` |
| `422 Webhook host blocked: loopback IP range` | Webhooks to localhost are refused (SSRF protection) | Use a public HTTPS URL on an allowlist; test locally with polling or `runs.join` |
| Second message gets 409 | `multitask_strategy="reject"` | Show "still answering", or choose `enqueue` / `interrupt` |
| Threads gone after a restart | `langgraph dev` keeps everything in memory | Use a deployment (Postgres) for anything that must persist |

## ⚠️ Production notes

> ⚠️ **Production note: auth is a list of attacker tests.** For every handler, call it with another customer's key, with no key, and with a customer key on a manager action. Part 3 and Part 4 are those tests; keep them in CI.

> ⚠️ **Production note: know what is metered.** On LangSmith Cloud a deployment's uptime is billed while it is live, and traces count against the plan. Delete dev deployments when you are done and set a usage limit.

> ⚠️ **Production note: `langgraph dev` is not a server for users.** It is single-process and in memory. Production means LangSmith Cloud, a standalone server with Postgres and Redis, or your own service.

## 🏋️ Exercises

1. **(Easy)** Make `enqueue`'s behaviour explicit for the desk: what does a client have to send to get `reject` instead, and which HTTP code must the UI handle?
2. **(Medium)** Add an `@auth.on.crons.create` handler that lets only managers create crons. Test it with the customer key.
3. **(Medium)** Use `RemoteGraph` as a node: build a two-node local graph where node 1 is `remote` and node 2 adds a signature to the last message.
4. **(Stretch)** With a Plus plan (optional, may cost money): deploy `apps/m12_agent_server` from your fork with *Deployments → New*, set `GROQ_API_KEY` and `MASTERY_SERVICE_MODE=live`, and point Agent Chat UI at it.

<details>
<summary><b>Show solutions</b></summary>

**1.** `multitask_strategy="reject"` on `runs.create`; the second request gets **409 Conflict**, so the UI shows "still answering" and keeps the text.

**2.**
```python
@auth.on.crons.create
async def managers_schedule(ctx, value):
    if "manager" not in ctx.permissions:
        raise Auth.exceptions.HTTPException(status_code=403, detail="managers only")
```
The more specific handler wins over `@auth.on`, so the customer's `crons.create` gets 403.

**3.**
```python
from langgraph.graph import START, StateGraph, MessagesState
def sign(state): return {"messages": [{"role": "assistant", "content": state["messages"][-1].content + " · Orbita"}]}
g = StateGraph(MessagesState).add_node("desk", remote).add_node("sign", sign)
g = g.add_edge(START, "desk").add_edge("desk", "sign").compile()
```

**4.** The deployment builds from `langgraph.json`; `dependencies: ["../.."]` must point inside the repo you connect. Delete the deployment afterwards: uptime is metered.
</details>

## 🔎 Docs drill (5 min)

In the double-texting docs, which strategy is the default, and which one also removes the first run's input?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Double texting](https://docs.langchain.com/langsmith/double-texting).
2. **Enqueue** is the default. **Rollback** reverts all progress *including the initial run input*.
3. Compare with the Part 5 table: `rollback` kept 1 user message.

</details>

## ✅ Checkpoint

- [ ] I can map threads, runs, assistants and crons onto our 12.04 service.
- [ ] `langgraph dev` served the desk from `langgraph.json`, with our auth.
- [ ] I saw 401, 404 and 403 for the wrong callers, and the server-set context.
- [ ] Only the manager could answer the refund approval.
- [ ] I can pick a double-texting strategy and say what each does to the first run.
- [ ] I know why webhooks to localhost are refused.
- [ ] I can explain why 12.09 ships our own service, and when I would pick LangSmith Deployment.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [langgraph-cli](https://pypi.org/project/langgraph-cli/) and [langgraph-api](https://pypi.org/project/langgraph-api/) on PyPI and the [pricing page](https://www.langchain.com/pricing), and log in [`docs/research-log.md`](../../docs/research-log.md): versions (2026-10-16: cli 0.4.36, api 0.16.1) and whether Plus still includes a free deployment.

## Summary

- The Agent Server gives threads, runs, a task queue, double texting, crons, webhooks, Studio and `/mcp` for any LangGraph graph.
- Custom auth filters by owner metadata; set identity on the server, and decide who may resume an interrupt.
- Double texting: reject (409), enqueue (default), interrupt (keep progress), rollback (drop it).
- On a free budget we ship our own service; the graph stays portable.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-09-ship-it.ipynb** → [`12-09-ship-it.ipynb`](12-09-ship-it.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)